# Vertical profiles of chemical species on the native cubed-sphere mesh

Time-mean day-side and night-side mean profiles of the mass fractions of the major gases and of the alkali metals in an HD 209458b simulation,
taken from the UGRID output (`lfric_diag_native*.nc`) without regridding.
The hemisphere means are area-weighted, computed with `aeolus.lfric.ugrid_spatial`, as in `Vertical-Profiles.ipynb`.

In [ ]:
import warnings

import iris
import matplotlib.pyplot as plt
import numpy as np
import paths
import seaborn as sns
from aeolus.calc import time_mean
from aeolus.lfric import ugrid_spatial
from aeolus.model import lfric
from aeolus.plot import add_custom_legend, figsave
from common import (
    CTRL_EXPERIMENTS as EXPERIMENTS,
    PA_TO_BAR,
    load_exp_days,
    setup_pressure_axis,
)

warnings.filterwarnings("ignore")
iris.FUTURE.date_microseconds = True

print(*[k for k in EXPERIMENTS if k.startswith("hd209")], sep="\n")

## Functions for UGRID cubes

In [ ]:
def _lonlat_to_xyz(lons, lats):
    """Convert longitudes and latitudes (degrees) to unit vectors."""
    lam, phi = np.deg2rad(np.asarray(lons, float)), np.deg2rad(np.asarray(lats, float))
    return np.stack(
        [np.cos(phi) * np.cos(lam), np.cos(phi) * np.sin(lam), np.sin(phi)], axis=-1
    )


def mesh_face_areas(cube, radius=1.0):
    """
    Areas of the (convex) faces of the mesh a cube is located on.

    Each face is split into triangles around its first node, and the area
    of each spherical triangle is found from its solid angle
    (Van Oosterom & Strackee, 1983).
    """
    mesh = cube.mesh
    face_nodes = np.asarray(mesh.face_node_connectivity.indices_by_location())
    node_xyz = _lonlat_to_xyz(
        mesh.node_coords.node_x.points, mesh.node_coords.node_y.points
    )
    corners = node_xyz[face_nodes]  # (n_faces, n_corners, 3)
    area = np.zeros(face_nodes.shape[0])
    a = corners[:, 0]
    for j in range(1, face_nodes.shape[1] - 1):
        b, c = corners[:, j], corners[:, j + 1]
        numer = np.abs(np.einsum("ij,ij->i", a, np.cross(b, c)))
        denom = (
            1
            + np.einsum("ij,ij->i", a, b)
            + np.einsum("ij,ij->i", b, c)
            + np.einsum("ij,ij->i", c, a)
        )
        area += 2 * np.arctan2(numer, denom)
    return area * radius**2


def hemisphere_mask(cube, side, substellar_lon=0.0):
    """
    Boolean mask of mesh faces on the day side or night side of a tidally locked planet.

    A face is on the day side if the cosine of the stellar zenith angle
    at its centre is positive, i.e. its longitude is within 90 degrees
    of the substellar longitude.
    """
    dlon = np.deg2rad(cube.coord(lfric.x).points - substellar_lon)
    if side == "day":
        return np.cos(dlon) > 0
    elif side == "night":
        return np.cos(dlon) < 0
    raise ValueError(f"side must be 'day' or 'night', got {side!r}")


def ugrid_hemisphere_mean(cube, side, substellar_lon=0.0, areas=None):
    """
    Area-weighted mean of a UGRID cube over the day-side or night-side hemisphere.

    The mean is over all mesh faces using `aeolus.lfric.ugrid_spatial`,
    with zero weights for the faces of the opposite hemisphere.
    """
    if areas is None:
        areas = mesh_face_areas(cube)
    weights = areas * hemisphere_mask(cube, side, substellar_lon=substellar_lon)
    (mesh_dim,) = cube.coord_dims(lfric.x)
    shape = [1] * cube.ndim
    shape[mesh_dim] = -1
    weights = np.broadcast_to(weights.reshape(shape), cube.shape)
    return ugrid_spatial(cube, "mean", weights=weights)

## Parameters

In [ ]:
EXP_KEY = "hd209_ctrl"
N_DAYS_MEAN = (
    200  # length of the averaging period (native output is written every 10 days)
)
# Hemisphere mean(s) to plot: "day", "night" or ("day", "night")
# (solid lines for the day side, dashed for the night side)
HEMISPHERE_MEAN = ("day", "night")
# Species to plot (as in the `mass_fraction_of_<species>_in_air` names) and their labels:
# the major gases, then the alkali metals
SPECIES = {
    "hydrogen": "H$_2$",
    "helium": "He",
    "water": "H$_2$O",
    "carbon_monoxide": "CO",
    "methane": "CH$_4$",
    "ammonia": "NH$_3$",
    "lithium": "Li",
    "sodium": "Na",
    "potassium": "K",
    "rubidium": "Rb",
    "cesium": "Cs",
}
PRES_LIM = (
    None  # (bottom, top) of the plot in bar, e.g. (200, 1e-5); None for the full range
)
SAVE_FIGS = True

EXP = EXPERIMENTS[EXP_KEY]
END_DAY = EXP.run_length
START_DAY = END_DAY - N_DAYS_MEAN
PERIOD_STR = f"days {START_DAY}-{END_DAY}" if N_DAYS_MEAN > 10 else f"day {END_DAY}"
if isinstance(HEMISPHERE_MEAN, str):
    HEMISPHERE_MEAN = (HEMISPHERE_MEAN,)
print(
    EXP.title, EXP.label, PERIOD_STR, f"substellar lon: {EXP.substellar_lon}", sep="\n"
)

## Load the data

In [ ]:
ds = load_exp_days(
    EXP_KEY,
    START_DAY,
    END_DAY,
    file_glob="lfric_diag_native*.nc",
    experiments=EXPERIMENTS,
)
pres = time_mean(ds.extract_cube("air_pressure"))
mass_fracs = {}
for species in SPECIES:
    cube = time_mean(ds.extract_cube(f"mass_fraction_of_{species}_in_air"))
    # Mass fractions and pressure are on the same (full) levels in the native output
    assert cube.coord(lfric.z) == pres.coord(lfric.z)
    mass_fracs[species] = cube
print(*[c.summary(shorten=True) for c in mass_fracs.values()], sep="\n")

In [ ]:
# Hemisphere means
areas = mesh_face_areas(pres)
print(f"Sum of face areas / 4pi = {areas.sum() / (4 * np.pi):.6f}")
hemi_means = {
    side: {
        "pres": ugrid_hemisphere_mean(pres, side, EXP.substellar_lon, areas=areas),
        **{
            species: ugrid_hemisphere_mean(cube, side, EXP.substellar_lon, areas=areas)
            for species, cube in mass_fracs.items()
        },
    }
    for side in HEMISPHERE_MEAN
}

## Plot the profiles

Each colour is a species, labelled above the top of its day-side profile
(labels too close to each other are moved apart); solid lines are the day-side means, dashed lines the night-side means.
The day-side and night-side means are averages of the mass fraction and of the pressure
in each model layer, i.e. on surfaces of constant height.

In [ ]:
KW_SIDE = {
    "day": {"ls": "-"},
    "night": {"ls": "--", "dash_capstyle": "round"},
}
SIDE_LABELS = {"day": "Day side", "night": "Night side"}


def species_colors(species):
    """Colours of the species, fixed by their order in `SPECIES` (up to 11 species)."""
    palette = [*sns.color_palette("tab10"), "k"]
    if len(species) > len(palette):
        raise ValueError(f"At most {len(palette)} species can be plotted")
    return dict(zip(species, palette))


def spread_positions(x, widths, pad=0.0, lo=0.0, hi=1.0):
    """
    Move the centres `x` of labels of given `widths` apart so that they do not overlap.

    Neighbouring labels are at least `pad` apart. Overlapping labels are merged
    into clusters, each centred on the mean of its original positions
    and kept within [lo, hi].
    """
    x, widths = np.asarray(x, float), np.asarray(widths, float)
    order = np.argsort(x)
    xs, ws = x[order], widths[order]
    # Clusters as lists of indices into the sorted positions
    clusters = [[i] for i in range(len(x))]

    def place(cl):
        """Centres of the labels in a cluster."""
        w = ws[cl]
        offsets = np.concatenate([[0], np.cumsum((w[:-1] + w[1:]) / 2 + pad)])
        shift = np.mean(xs[cl] - offsets)
        shift = min(max(shift, lo + w[0] / 2), hi - offsets[-1] - w[-1] / 2)
        return shift + offsets

    merged = True
    while merged:
        merged = False
        for k in range(len(clusters) - 1):
            right = place(clusters[k])[-1] + ws[clusters[k][-1]] / 2
            left = place(clusters[k + 1])[0] - ws[clusters[k + 1][0]] / 2
            if left - right < pad:
                clusters[k : k + 2] = [clusters[k] + clusters[k + 1]]
                merged = True
                break
    new = np.empty_like(x)
    for cl in clusters:
        new[order[cl]] = place(cl)
    return new


def label_profile_tops(ax, profiles, colors, labels, pad_pt=3, gap_pt=2):
    """
    Label profiles just above the top of the axes, at their mass fraction at the top pressure.

    `profiles` maps a species to a (mass fraction, pressure) pair of arrays.
    The axes must have a logarithmic x axis and its limits already set.
    Labels that would overlap are moved apart, at least `pad_pt` points from each other.
    """
    p_top = min(ax.get_ylim())
    log_xlim = np.log10(ax.get_xlim())
    texts, frac = [], []
    for species, (mf, pres) in profiles.items():
        isort = np.argsort(pres)
        x_top = np.interp(np.log10(p_top), np.log10(pres[isort]), np.log10(mf[isort]))
        x_frac = ((x_top - log_xlim[0]) / np.diff(log_xlim)).item()
        if not 0 <= x_frac <= 1:
            continue
        frac.append(x_frac)
        texts.append(
            ax.annotate(
                labels[species],
                xy=(x_frac, 1),
                xycoords="axes fraction",
                xytext=(0, gap_pt),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize="medium",
                color=colors[species],
                annotation_clip=False,
            )
        )
    # Widths of the labels as fractions of the axes width
    renderer = ax.figure.canvas.get_renderer()
    ax_width_px = ax.get_window_extent(renderer).width
    widths = [t.get_window_extent(renderer).width / ax_width_px for t in texts]
    pad = pad_pt * ax.figure.dpi / 72 / ax_width_px
    for t, x_new in zip(texts, spread_positions(frac, widths, pad=pad)):
        t.xy = (x_new, 1)


def plot_species_profiles(mf_lim=None, pres_lim=PRES_LIM):
    fig, ax = plt.subplots(figsize=(4, 3), layout="constrained")
    colors = species_colors(SPECIES)
    for side, means in hemi_means.items():
        p_mean = means["pres"].data * PA_TO_BAR
        for species in SPECIES:
            ax.plot(
                means[species].data,
                p_mean,
                color=colors[species],
                lw=1.5,
                **KW_SIDE[side],
            )
    setup_pressure_axis(ax, lim=pres_lim)
    ax.set_xscale("log")
    ax.set_xlim(mf_lim)
    ax.set_xlabel("Mass fraction [$kg$ $kg^{-1}$]")
    sns.despine(ax=ax, offset=5)
    # Species labels at the top of their day-side profiles
    # (or of the first hemisphere plotted, if the day side is not)
    side_lab = "day" if "day" in hemi_means else next(iter(hemi_means))
    p_lab = hemi_means[side_lab]["pres"].data * PA_TO_BAR
    label_profile_tops(
        ax,
        {sp: (hemi_means[side_lab][sp].data, p_lab) for sp in SPECIES},
        colors,
        SPECIES,
    )
    # Legend of the hemispheres (line styles)
    add_custom_legend(
        ax,
        {
            SIDE_LABELS[side]: {"color": "k", "lw": 1.5, **KW_SIDE[side]}
            for side in hemi_means
        },
        loc="center left",
        fontsize="small",
        frameon=False,
    )
    ax.set_title(f"{EXP.title} | {PERIOD_STR}", size="medium", pad=16)
    if SAVE_FIGS:
        figsave(
            fig,
            paths.drafts
            / "profiles"
            / f"{EXP_KEY}__mass_fraction_profiles__{'_'.join(hemi_means)}__days_{START_DAY}_{END_DAY}",
        )
    return fig


plot_species_profiles(mf_lim=(1e-13, 1e0), pres_lim=(200, 1e-5));